# 04 — GGUF Conversion & Quantization
Merges the LoRA adapter into the base model, converts to GGUF format, 
and quantizes to Q4_K_M for efficient offline inference on CPU and Android.

**Input:** Merged model — `souravchandra01/V2-TigerLLM-Medical-BN`  
**Output:** `swasthya-1b-q4km.gguf` (~700MB) pushed to HuggingFace

## Installing Dependencies & Authentication

In [1]:
!pip install -q transformers peft accelerate
!pip install -q huggingface_hub
!pip install -q torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.5 MB/s eta 0:00:00


In [8]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully!")

Logged in successfully!


## Downloading Merged Model from HuggingFace

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="souravchandra01/V2-TigerLLM-Medical-BN",
    local_dir="/content/swasthya_model"
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/content/swasthya_model'

In [4]:
model_path = "/content/swasthya_model"

## Loading Merged Model & Tokenizer

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

Loading tokenizer...


In [6]:
import torch

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype = torch.bfloat16,
    device_map = "auto"
)


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

## Vocab Size Mismatch Fix
TigerLLM's tokenizer includes an extra vision token `<image_soft_token>` 
(id: 262144) not present in the base model weights — causing a mismatch 
during GGUF conversion. Fixed by updating `config.json` vocab_size to 262145.

In [8]:
print(f"Model vocab size: {model.config.vocab_size}")
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Embedding shape: {model.model.embed_tokens.weight.shape}")

# Resize model embeddings to match tokenizer
model.resize_token_embeddings(len(tokenizer))

print(f"New embedding shape: {model.model.embed_tokens.weight.shape}")

# Save fixed model
model.save_pretrained('/content/swasthya_model-fixed')
tokenizer.save_pretrained('/content/swasthya_model-fixed')
print("Saved fixed model!")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model vocab size: 262144
Tokenizer vocab size: 262145
Embedding shape: torch.Size([262144, 1152])


[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


New embedding shape: torch.Size([262145, 1152])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fixed model!


In [1]:
import json

with open('/content/swasthya_model/config.json', 'r') as f:
    config = json.load(f)

# Update to match actual tokenizer
config['vocab_size'] = 262145

with open('/content/swasthya_model/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Fixed vocab size to 262145")

Fixed vocab size to 262145


## Verifying Merged Model

In [ ]:
#system prompt
SYSTEM_PROMPT = """তুমি স্বাস্থ্যা, একটি স্বাস্থ্য সহায়ক।

গুরুত্বপূর্ণ নিয়ম:

* ব্যবহারকারী যে ভাষায় প্রশ্ন করবে, সেই ভাষাতেই উত্তর দেবে
* ব্যবহারকারী ইংরেজিতে লিখলে ইংরেজিতে উত্তর দাও।
* রোগ নির্ণয় নিশ্চিতভাবে করবে না।
* সাধারণ স্বাস্থ্য পরামর্শ দেবে।
* জরুরি লক্ষণ থাকলে দ্রুত ডাক্তার বা হাসপাতালে যেতে বলবে।

"""

In [ ]:
def generate_response(prompt, max_new_tokens=200):

    messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
test_prompts = [
    "I have fever since 2 days and headache. What should I do?",
    "আমার দুই দিন ধরে জ্বর এবং মাথাব্যথা। আমার কী করা উচিত?",
    "My child has cough and cold since 3 days.",
    "আমার পেটে ব্যথা হচ্ছে এবং বমি বমি ভাব লাগছে।",
]

for prompt in test_prompts:
    response = generate_response(prompt)
    print(f"PROMPT: {prompt}")
    print(f"RESPONSE: {response}")
    print()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: I have fever since 2 days and headache. What should I do?
RESPONSE: Hello, I am here to help you with your query. The symptoms you are describing can be caused by a variety of reasons. It is not possible to diagnose the cause without more information. You should consult a doctor for further evaluation. Hope this helps. Let me know if you have any other questions.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: আমার দুই দিন ধরে জ্বর এবং মাথাব্যথা। আমার কী করা উচিত?
RESPONSE: হ্যালো, আপনার উপসর্গগুলো সাধারণত ঠান্ডা লাগা বা ফ্লু-এর কারণে হতে পারে। যদি জ্বর বেশি থাকে (৩৯ ডিগ্রি সেলসিয়াস বা তার উপরে), মাথা ব্যথা তীব্র হয়, অথবা অন্য কোনো গুরুতর উপসর্গ দেখা দেয়, তাহলে অবশ্যই একজন চিকিৎসকের কাছে যান। নিজে থেকে চিকিৎসা শুরু করবেন না।



[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: My child has cough and cold since 3 days.
RESPONSE: Hi, I understand your concern. Cough and cold are common symptoms of a viral infection. It is recommended to rest, drink plenty of fluids, and take over-the-counter medications like ibuprofen or acetaminophen for pain and fever. If the symptoms persist or worsen, please seek medical advice from a doctor. Hope this helps. Let me know if you have any further questions.

PROMPT: আমার পেটে ব্যথা হচ্ছে এবং বমি বমি ভাব লাগছে।
RESPONSE: হ্যালো, আপনার পেটের সমস্যার কারণ হতে পারে বিভিন্ন কারণে যেমন খাদ্যনালীর সংক্রমণ, গ্যাস্ট্রিক আলসার, ম্যালেরিয়া ইত্যাদি। সঠিক কারণ নির্ধারণের জন্য একজন চিকিৎসকের কাছে যাওয়া উচিত।



## Setting Up llama.cpp

In [9]:
import os
os.chdir('/content')

!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp --depth 1
%cd llama.cpp
!pip install -r requirements.txt

Cloning into 'llama.cpp'...
remote: Enumerating objects: 3274, done.
remote: Counting objects: 100% (3274/3274), done.
remote: Compressing objects: 100% (2610/2610), done.
remote: Total 3274 (delta 658), reused 2239 (delta 585), pack-reused 0 (from 0)
Receiving objects: 100% (3274/3274), 32.54 MiB | 16.51 MiB/s, done.
Resolving deltas: 100% (658/658), done.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━

In [4]:
%cd /content/llama.cpp

/content/llama.cpp


## Converting to GGUF (F16)

In [5]:
!python convert_hf_to_gguf.py \
    /content/swasthya_model-fixed \
    --outfile /content/swasthya-1b-medical-f16.gguf \
    --outtype f16

print("Converted to GGUF!")

INFO:hf-to-gguf:Loading model: swasthya_model-fixed
INFO:hf-to-gguf:Model architecture: Gemma3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> F16, shape = {1152, 262145}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.bfloat16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.bfloat16 --> F16, shape = {6912, 1152}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.bfloat16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.post_ffw_norm.weight,        torch.bfloat16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,             torch.bfloat16 --> F3

## Quantizing to Q4_K_M

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Quantize to Q4_K_M
!cmake -B build
!cmake --build build --config Release

!./build/bin/llama-quantize \
    /content/swasthya-1b-medical-f16.gguf \
    /content/drive/MyDrive/swasthya/swasthya-1b-medical-q4km.gguf \
    Q4_K_M

print("Quantization complete!")

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

## Size Comparison

In [7]:
import os

f16_size = os.path.getsize("/content/swasthya-1b-medical-f16.gguf") / 1024**3
q4_size = os.path.getsize("/content/drive/MyDrive/swasthya/swasthya-1b-medical-q4km.gguf") / 1024**3

print(f"F16 GGUF size  : {f16_size:.2f} GB")
print(f"Q4_K_M size    : {q4_size:.2f} GB")
print(f"Size reduction : {(1 - q4_size/f16_size)*100:.1f}%")

F16 GGUF size  : 1.88 GB
Q4_K_M size    : 0.76 GB
Size reduction : 59.6%


## Pushing to HuggingFace

In [9]:
from huggingface_hub import HfApi

api = HfApi()

# Push Q4 GGUF
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/swasthya/swasthya-1b-medical-q4km.gguf",
    path_in_repo="swasthya-1b-q4km.gguf",
    repo_id="souravchandra01/swasthya-1b-gguf",
    repo_type="model",
    token=hf_token
)

print("GGUF model pushed to HuggingFace!")
print("Available at: https://huggingface.co/souravchandra01/swasthya-1b-gguf")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...thya-1b-medical-q4km.gguf:   1%|          | 8.06MB /  814MB            

GGUF model pushed to HuggingFace!
Available at: https://huggingface.co/souravchandra01/swasthya-1b-gguf
